# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Prabhaditya003/FlyRank.ai-internship-work-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Signal check 1 -- CTR vs. position** (behind the CTR-fix logic). Hypothesis: pages that
already rank well should get meaningfully higher CTR than pages ranking poorly. Bucketed by
`position_tier` (excluding the 1,205 rows where `avg_position == 0` -- that means *no ranking
data*, not rank zero, per the data-dictionary gotcha), mean CTR falls monotonically: top_3
(2.76%) -> page_1 (0.65%) -> striking (0.32%) -> page_3_5 (0.22%) -> deep (0.15%), each bucket
backed by 1,100+ rows. **Verdict: CONFIRMED.**

**Signal check 2 -- Volume** (behind the quick-win logic). Hypothesis: high-impression content
is where a fix pays off biggest, and should skew toward the "good-but-not-top" position range
quick-wins target. Bucketed by `impression_tier`, neither mean CTR nor mean position moves
monotonically with volume -- the `low` tier's high mean CTR (0.94%) is a small-denominator
artifact (one click on a handful of impressions swings the percentage wildly), and
`moderate`-volume content is actually positioned *worse* (18.7) than `low`-volume content
(16.1). **Verdict: MIXED.** Volume alone doesn't cleanly separate opportunity -- it has to be
paired with a position-based signal rather than trusted by itself. That's the useful negative:
weight the score by impressions, but don't rank by impressions alone.

**My rule, in plain words:** a page is worth reviewing if it's already sitting in a fixable
position (not already top 3, not buried past page 5) and getting decent search volume, but its
CTR is below what pages in that same position tier normally get -- because that gap points to an
on-page fix (title/meta), not a ranking problem.

**Reason code:** `ctr_below_tier_benchmark` (one code, for every row the rule flags).

**Action label:** `review_meta_title` (unflagged rows get `no_action` / `not_flagged`).


In [1]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

has_pos = df['avg_position'] > 0
order_pos = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
sig1 = (df.loc[has_pos]
          .groupby('position_tier')['ctr']
          .agg(mean_ctr='mean', median_ctr='median', n='count')
          .reindex(order_pos))
print("SIGNAL 1 -- CTR by position tier (avg_position>0 only; 0 = 'no data', not rank 0)")
print(sig1.round(3))
print(f"rows excluded as no-data (avg_position == 0): {(df['avg_position']==0).sum()}")
print("verdict: CONFIRMED -- mean CTR falls monotonically as position tier worsens")

print()

order_imp = ['low', 'moderate', 'good', 'excellent']
sig2 = (df.groupby('impression_tier')
          .agg(mean_ctr=('ctr', 'mean'), mean_position=('avg_position', 'mean'), n=('ctr', 'size'))
          .reindex(order_imp))
print("SIGNAL 2 -- CTR and avg_position by impression (volume) tier")
print(sig2.round(3))
print("verdict: MIXED -- neither column moves monotonically with volume;")
print("'low' bucket's high mean CTR is a small-denominator artifact, and")
print("'moderate' volume is positioned *worse* than 'low' volume on average")


SIGNAL 1 -- CTR by position tier (avg_position>0 only; 0 = 'no data', not rank 0)
               mean_ctr  median_ctr      n
position_tier                             
top_3             2.764        0.00   1116
page_1            0.652        0.16  11814
striking          0.323        0.11   7304
page_3_5          0.222        0.03   7242
deep              0.150        0.00   1319
rows excluded as no-data (avg_position == 0): 1205
verdict: CONFIRMED -- mean CTR falls monotonically as position tier worsens

SIGNAL 2 -- CTR and avg_position by impression (volume) tier
                 mean_ctr  mean_position      n
impression_tier                                
low                 0.937         16.125  11248
moderate            0.212         18.748  10469
good                0.308         13.853   7205
excellent           0.313         11.889   1078
verdict: MIXED -- neither column moves monotonically with volume;
'low' bucket's high mean CTR is a small-denominator artifact, and
'moderat

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Score = `ctr_gap x impressions_90d`, gated to zero unless all three plain conditions hold:
fixable position, decent volume, and an actual CTR shortfall vs. the page's own tier benchmark.
No fitted weights -- just addition/multiplication of readable conditions, per the baseline
skill.


In [2]:
import pandas as pd

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

has_pos = df['avg_position'] > 0
tier_benchmark = df.loc[has_pos].groupby('position_tier')['ctr'].mean()

df['tier_benchmark_ctr'] = df['position_tier'].map(tier_benchmark)
df.loc[~has_pos, 'tier_benchmark_ctr'] = pd.NA          # no benchmark for no-data rows
df['ctr_gap'] = df['tier_benchmark_ctr'] - df['ctr']     # positive = underperforming its tier

fixable_position = df['position_tier'].isin(['page_1', 'striking', 'page_3_5']) & has_pos
decent_volume    = df['impression_tier'].isin(['moderate', 'good', 'excellent'])
underperforming  = df['ctr_gap'] > 0

eligible = fixable_position & decent_volume & underperforming   # readable on purpose

df['score'] = 0.0
df.loc[eligible, 'score'] = df.loc[eligible, 'ctr_gap'] * df.loc[eligible, 'impressions_90d']

df['reason_code'] = 'not_flagged'
df.loc[eligible, 'reason_code'] = 'ctr_below_tier_benchmark'   # ONE reason code

df['action'] = 'no_action'
df.loc[eligible, 'action'] = 'review_meta_title'                # ONE action label

ranked = df.sort_values('score', ascending=False).reset_index(drop=True)

out_cols = ['content_id', 'client_id', 'position_tier', 'avg_position', 'impression_tier',
            'impressions_90d', 'ctr', 'tier_benchmark_ctr', 'ctr_gap', 'score',
            'reason_code', 'action']
ranked_out = ranked[out_cols]
ranked_out.to_csv('work/outputs/baseline_action_score.csv', index=False)

print(f"flagged (score > 0): {eligible.sum():,} of {len(df):,} rows ({100*eligible.mean():.1f}%)")
print(f"reason codes -> {ranked_out['reason_code'].value_counts().to_dict()}")
print(f"wrote work/outputs/baseline_action_score.csv  ({len(ranked_out):,} rows)")


flagged (score > 0): 14,336 of 30,000 rows (47.8%)
reason codes -> {'not_flagged': 15664, 'ctr_below_tier_benchmark': 14336}
wrote work/outputs/baseline_action_score.csv  (30,000 rows)


## 3. Top-10 review

*For each of your top ten, one line each -- the action, why it's there, and what would make it
wrong.*


In [3]:
import pandas as pd

ranked_out = pd.read_csv('work/outputs/baseline_action_score.csv')
top10 = ranked_out.head(10)

pd.set_option('display.width', 160)
print(top10.to_string(index=True))
print()

for i, row in top10.iterrows():
    if row['ctr'] == 0.0:
        wrong_if = "this is a tracking/attribution gap (zero recorded clicks on 200k+ impressions is unusual) rather than a genuine on-page problem"
    elif row['avg_position'] > 8:
        wrong_if = "it's sitting near the bottom of page_1 -- close enough to 'striking' that the benchmark it's judged against may be too generous"
    else:
        wrong_if = "the low CTR reflects a mismatched query/intent (page ranks for a term it doesn't actually answer) rather than a fixable title or meta description"
    print(
        f"{i+1:>2}. {row['action']} on {row['content_id']} ({row['client_id']}) -- "
        f"reason: {row['reason_code']}; CTR {row['ctr']:.2f}% vs. {row['position_tier']} "
        f"benchmark {row['tier_benchmark_ctr']:.2f}% on {row['impressions_90d']:,} impressions. "
        f"Wrong if: {wrong_if}."
    )


             content_id          client_id position_tier  avg_position impression_tier  impressions_90d   ctr  tier_benchmark_ctr   ctr_gap          score               reason_code             action
0  content_5fe46e04994d  client_4e07408562        page_1           4.2       excellent           517715  0.14            0.652467  0.512467  265311.627747  ctr_below_tier_benchmark  review_meta_title
1  content_aaef01a50def  client_19581e27de        page_1           5.4       excellent           517109  0.25            0.652467  0.402467  208119.083008  ctr_below_tier_benchmark  review_meta_title
2  content_36ff89c8214e  client_19581e27de        page_1           7.3       excellent           295097  0.05            0.652467  0.602467  177786.075959  ctr_below_tier_benchmark  review_meta_title
3  content_1a9e894be2e2  client_19581e27de        page_1           4.0       excellent           416180  0.23            0.652467  0.422467  175822.135060  ctr_below_tier_benchmark  review_meta_title


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak pick:** 11 of the top 20 queue entries belong to a single client
(`client_19581e27de`). The score has no per-client cap or normalization, so one client with
several huge-impression pages crowds out everyone else -- a real reviewer would want the queue
diversified per client before acting on it, not just sorted globally by score.

**Leakage check:** the score only touches `position_tier`, `avg_position`, `impression_tier`,
`impressions_90d`, and `ctr` -- never `trend_direction` / `trend_pct` (the label-trap columns).
`content_id` / `client_id` are used for identification only, never as score inputs. This CSV is
a single 90-day snapshot (no `report_date` panel), so there's no future-window column to leak
from here -- but the `*_last_30d` / `*_prev_30d` split columns were left out entirely on
principle rather than mixed in without checking their alignment.


In [4]:
import pandas as pd

ranked_out = pd.read_csv('work/outputs/baseline_action_score.csv')

print("Client concentration in the top 20 flagged rows:")
print(ranked_out.head(20)['client_id'].value_counts())
print()

score_input_cols = {'position_tier', 'avg_position', 'impression_tier', 'impressions_90d', 'ctr'}
label_trap_cols = {'trend_direction', 'trend_pct', 'is_declining_label'}
id_cols = {'content_id', 'client_id'}

print("Score inputs:", sorted(score_input_cols))
print("Label-trap columns touched by scoring logic:", score_input_cols & label_trap_cols, "(must be empty)")
print("ID columns used only for grouping/identification, never scored:",
      id_cols.isdisjoint(score_input_cols))
print("Dataset is a single 90-day snapshot (no report_date panel), so no future-window column exists to leak from;")
print("the *_last_30d / *_prev_30d split columns were left out entirely rather than mixed in unchecked.")


Client concentration in the top 20 flagged rows:
client_id
client_19581e27de    11
client_f369cb89fc     5
client_4e07408562     2
client_349c41201b     2
Name: count, dtype: int64

Score inputs: ['avg_position', 'ctr', 'impression_tier', 'impressions_90d', 'position_tier']
Label-trap columns touched by scoring logic: set() (must be empty)
ID columns used only for grouping/identification, never scored: True
Dataset is a single 90-day snapshot (no report_date panel), so no future-window column exists to leak from;
the *_last_30d / *_prev_30d split columns were left out entirely rather than mixed in unchecked.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled -- markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime -> Run all)
- [x] No client names, URLs, or private queries anywhere (client/content IDs are pseudonyms from the starter dataset)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` -- then submit your repo URL on the card. Done.
